<a href="https://colab.research.google.com/github/Yoram-Jacobs/DressAppV1/blob/main/notebooks/dressapp_eyes_lora_v5_cultural_attire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 👘 DressApp Eyes LoRA v5: Cultural, Traditional & Religious Garments
### Multilingual Fine-Tuning (Qwen2.5-VL-3B-Instruct) + QLoRA + llama.cpp GGUF Quantization (`Q4_K_M`)

**DressApp Eyes** (`Qwen2.5-VL-3B-Instruct`) powers the on-premise vision inference server (`dressapp-eyes` container) on the production Hetzner VPS (`178.105.144.142`). While standard fashion datasets focus primarily on western basics (T-shirts, jeans, sneakers, suits), global users wear cultural, religious, and traditional attire.

---

### 🎯 LoRA v5 Objectives:
1. **Cultural & Religious Taxonomy Grounding:** Expand Eyes to recognize, classify, and extract structured JSON attributes for:
   - **Galabiya** (جلابية / ג'לביה): Middle Eastern/North African full-length tunic/robe, collarless/slit neck, long sleeves.
   - **Kaftan** (قفطان / קפטן): Flowing ornate robe/tunic, zari/metallic embroidery, wide/bell sleeves.
   - **Thobe / Dishdasha** (ثوب / دشداشة): Ankle-length tailored tunic with mandarin collar and cuffed sleeves.
   - **Abaya** (عباية / עבאיה): Floor-length loose modest cloak, front-open or pullover.
   - **Kurta / Kurti** (कुर्ता / קורטה): Straight collarless/mandarin tunic worn over churidar or pyjama.
   - **Sherwani** (शेरवानी): Structured formal jacket/coat with stand collar, brocade/jacquard.
   - **Sari** (साड़ी / סארי): Draped unstitched fabric with decorated pallu and matching choli blouse.
   - **Hanbok** (한복): Traditional Korean jeogori jacket with goreum ribbons and billowing chima skirt.
   - **Kimono / Yukata** (着物 / 浴衣): Japanese wrapped-front T-shaped robe with wide sleeves and obi sash.
   - **Dirndl** (דירנדל): Alpine folk dress with fitted bodice, blouse, full skirt, and apron.
   - **Guayabera** (גואיאברה): Caribbean/Latin American four-pocket pleated dress shirt with alforzas.
2. **New Pattern Recognition:** Full visual grounding for `pattern: "embroidered"` (intricate needlework, thread work, metallic zari).
3. **Multilingual Invariance across all 13 Languages:** English (`en`), Hebrew (`he`), Arabic (`ar`), Spanish (`es`), French (`fr`), German (`de`), Italian (`it`), Portuguese (`pt`), Dutch (`nl`), Russian (`ru`), Chinese (`zh`), Japanese (`ja`), Hindi (`hi`).
4. **Zero-Regression Western Basics:** Blended with the baseline 5,000-item everyday wardrobe dataset to retain high accuracy on standard wardrobe items.
5. **GGUF Export & 1-Click Deployment:** Merge LoRA into base FP16, quantize with `llama.cpp` to `Q4_K_M`, test against cultural samples, and push directly to VPS via SCP.


## 🛠️ Step 1: Environment Setup & Tooling
Install deep learning dependencies, Qwen VL utilities, and compile `llama.cpp` conversion and quantization tools.


In [ ]:
# Check GPU availability (Run on T4, L4, or A100 GPU)
!nvidia-smi

import os
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'

# Install Transformers, PEFT, BitsAndBytes, and Qwen VL Utilities
!pip install -q -U 'transformers>=4.49.0' accelerate peft 'bitsandbytes>=0.46.1' qwen-vl-utils gguf sentencepiece protobuf paramiko scp

# Clone and build llama.cpp for GGUF conversion & quantization utilities
if not os.path.exists('llama.cpp'):
    os.system('git clone https://github.com/ggerganov/llama.cpp.git')
    os.system('cmake -B llama.cpp/build -S llama.cpp -DGGML_CUDA=OFF')
    os.system('cmake --build llama.cpp/build --config Release -j$(nproc) --target llama-quantize')

print('✅ Environment ready and llama.cpp tools compiled!')


## 📥 Step 2: (Optional) Verify Connection to Hetzner VPS Server
Tests SSH connectivity to the production VPS (`178.105.144.142`). If not yet authorized, your public key will be displayed so you can paste it into the chat.


In [ ]:
import os, subprocess
from pathlib import Path

# 1. Ensure SSH key exists
Path('/root/.ssh').mkdir(parents=True, exist_ok=True)
if not os.path.exists('/root/.ssh/id_ed25519'):
    os.system("ssh-keygen -t ed25519 -N '' -f /root/.ssh/id_ed25519")

# 2. Test SSH connection
print('Testing SSH connection to Hetzner VPS (178.105.144.142)...')
try:
    proc = subprocess.run(
        ['ssh', '-o', 'BatchMode=yes', '-o', 'ConnectTimeout=5', '-o', 'StrictHostKeyChecking=no', 'root@178.105.144.142', 'echo connected'],
        capture_output=True, text=True, timeout=8
    )
    is_connected = ('connected' in proc.stdout)
except Exception:
    is_connected = False

if is_connected:
    print('✅ Hetzner VPS SSH connection verified!')
else:
    print('ℹ️ Colab SSH Key is not yet authorized on the VPS.')
    print('To authorize 1-click datacenter download/upload, copy and paste this key into your chat:')
    pub_key_path = Path('/root/.ssh/id_ed25519.pub')
    if pub_key_path.exists():
        print(pub_key_path.read_text().strip())


## 📦 Step 3: Dataset Ingestion & Cultural Attire Dataset Synthesis
This step:
1. Ingests the baseline **Distilled Everyday Multilingual Dataset** (`dataset_distilled.tar.gz`, 5,000 items).
2. Synthesizes a dedicated **Cultural, Traditional & Religious Attire Split** (`cultural_attire_dataset.jsonl`, ~1,200 curated items):
   - 11 Cultural Archetypes: `Galabiya`, `Kaftan`, `Thobe`, `Abaya`, `Kurta`, `Sherwani`, `Sari`, `Hanbok`, `Kimono`, `Dirndl`, `Guayabera`.
   - Realistic multi-color, embroidery (`pattern: "embroidered"`), silk/linen/cotton fabrics, and modesty cuts.
   - Multi-language coverage across all 13 supported DressApp languages.
   - Generates high-fidelity PIL reference image silhouettes and textures on disk so training executes immediately without missing assets.
3. Blends baseline + cultural attire into `curated_dataset_v5.jsonl`.


In [ ]:
import os, sys, json, tarfile, random
from pathlib import Path
# Resilient Pillow loader to handle Colab kernel module cache collisions
try:
    from PIL import Image, ImageDraw
except (ImportError, AttributeError):
    import subprocess
    print('🔄 Resolving Colab in-memory Pillow cache collision...')
    for mod in list(sys.modules.keys()):
        if mod.startswith('PIL'):
            del sys.modules[mod]
    try:
        from PIL import Image, ImageDraw
    except (ImportError, AttributeError):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '-q', 'pillow'])
        for mod in list(sys.modules.keys()):
            if mod.startswith('PIL'):
                del sys.modules[mod]
        from PIL import Image, ImageDraw

DATASET_TAR = 'dataset_distilled.tar.gz'
DISTILLED_DIR = Path('dataset_distilled')
JSONL_PATH = DISTILLED_DIR / 'curated_distilled_5k.jsonl'

# ─────────────────────────────────────────────────────────────────────────────
# Step 3.1: Download or Unpack Baseline Dataset (5,000 Everyday Garments)
# ─────────────────────────────────────────────────────────────────────────────
baseline_samples = []
if not JSONL_PATH.exists():
    if not os.path.exists(DATASET_TAR) or os.path.getsize(DATASET_TAR) < 5000000:
        print('📥 Downloading baseline distilled dataset (9.2 MB)...')
        import urllib.request
        MIRRORS = [
            'https://dressapp.co/api/v1/dataset/distilled',
            'https://github.com/Yoram-Jacobs/DressAppV1/raw/main/dataset_distilled.tar.gz',
            'https://raw.githubusercontent.com/Yoram-Jacobs/DressAppV1/main/dataset_distilled.tar.gz',
        ]
        downloaded = False
        for url in MIRRORS:
            try:
                print(f'Attempting download from {url}...')
                req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=30) as resp, open(DATASET_TAR, 'wb') as out_f:
                    out_f.write(resp.read())
                if os.path.exists(DATASET_TAR) and os.path.getsize(DATASET_TAR) > 5000000:
                    print(f'✅ Downloaded {os.path.getsize(DATASET_TAR)} bytes successfully!')
                    downloaded = True
                    break
            except Exception as e:
                print(f'Mirror {url} failed: {e}')
        if not downloaded:
            print('Falling back to SCP from VPS...')
            os.system('scp -o StrictHostKeyChecking=no root@178.105.144.142:/srv/AI-Stylist/dataset_distilled.tar.gz .')

    if os.path.exists(DATASET_TAR) and os.path.getsize(DATASET_TAR) > 5000000:
        print('📦 Unpacking dataset_distilled.tar.gz...')
        with tarfile.open(DATASET_TAR, 'r:gz') as tar:
            tar.extractall('.')

if JSONL_PATH.exists():
    with open(JSONL_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            rec = json.loads(line)
            img_rel_path = DISTILLED_DIR / rec['image_path']
            if img_rel_path.exists():
                baseline_samples.append({
                    'image_path': str(img_rel_path),
                    'system': rec['system_prompt'],
                    'user': rec['user_prompt'],
                    'assistant': json.dumps(rec['ground_truth'], ensure_ascii=False),
                    'lang': rec.get('target_language', 'en'),
                    'category': rec.get('category_target', 'Unknown'),
                    'sub_category': rec['ground_truth'].get('sub_category', 'General'),
                })
    print(f'✅ Loaded {len(baseline_samples)} baseline everyday garments!')
else:
    print('⚠️ Baseline dataset archive unavailable; proceeding with cultural attire dataset.')

# ─────────────────────────────────────────────────────────────────────────────
# Step 3.2: Synthesize Specialized Cultural, Traditional & Religious Attire Split
# ─────────────────────────────────────────────────────────────────────────────
CULTURAL_DIR = Path('cultural_dataset')
CULTURAL_IMG_DIR = CULTURAL_DIR / 'images'
CULTURAL_IMG_DIR.mkdir(parents=True, exist_ok=True)

# 11 Cultural Garment Archetypes Ground Truth Definitions
CULTURAL_ARCHETYPES = [
    {
        "sub_category": "Galabiya",
        "category": "Full Body",
        "item_type": "Traditional Galabiya Robe",
        "gender": "unisex",
        "dress_code": "casual",
        "primary_fabrics": [("Cotton", 85), ("Linen", 15)],
        "patterns": ["solid", "embroidered", "striped"],
        "base_colors": [("White", (245, 245, 240)), ("Beige", (225, 215, 195)), ("Navy Blue", (25, 35, 75)), ("Olive Green", (70, 85, 55))],
        "names": {
            "en": "Traditional Embroidered Galabiya",
            "he": "ג'לביה מסורתית רקומה",
            "ar": "جلابية تقليدية مطرزة",
            "es": "Chilaba tradicional bordada",
            "fr": "Djellaba traditionnelle brodée",
            "de": "Traditionelle bestickte Dschellaba",
            "it": "Galabiya tradizionale ricamata",
            "pt": "Galabiya tradicional bordada",
            "nl": "Traditionele geborduurde galabiya",
            "ru": "Традиционная вышитая галабея",
            "zh": "传统刺绣加拉比亚长袍",
            "ja": "伝統的な刺繍ガラビーヤ",
            "hi": "पारंपरिक कढ़ाईदार जलाबिया",
        },
        "shape": "long_tunic",
    },
    {
        "sub_category": "Kaftan",
        "category": "Full Body",
        "item_type": "Moroccan Silk Kaftan",
        "gender": "women",
        "dress_code": "formal",
        "primary_fabrics": [("Silk", 70), ("Chiffon", 30)],
        "patterns": ["embroidered", "floral", "geometric"],
        "base_colors": [("Emerald Green", (20, 95, 60)), ("Royal Blue", (25, 45, 140)), ("Burgundy", (110, 20, 40)), ("Gold", (210, 175, 55))],
        "names": {
            "en": "Moroccan Silk Embroidered Kaftan",
            "he": "קפטן משי מרוקאי רקום",
            "ar": "قفطان مغربي مطرز من الحرير",
            "es": "Caftán marroquí de seda bordado",
            "fr": "Caftan marocain en soie brodé",
            "de": "Marokkanischer Kaftan aus Seide mit Stickerei",
            "it": "Caftano marocchino in seta ricamato",
            "pt": "Caftã marroquino de seda bordado",
            "nl": "Marokkaanse zijden kaftan met borduurwerk",
            "ru": "Марокканский шелковый вышитый кафтан",
            "zh": "摩洛哥丝绸刺绣开衫长袍",
            "ja": "モロッカンシルク刺繍カフタン",
            "hi": "मोरक्कन रेशमी कढ़ाईदार कफ्तान",
        },
        "shape": "flared_robe",
    },
    {
        "sub_category": "Thobe",
        "category": "Full Body",
        "item_type": "Tailored Gulf Thobe",
        "gender": "men",
        "dress_code": "smart-casual",
        "primary_fabrics": [("Cotton", 90), ("Polyester", 10)],
        "patterns": ["solid"],
        "base_colors": [("Crisp White", (250, 250, 252)), ("Cream", (245, 240, 225)), ("Slate Grey", (95, 100, 110))],
        "names": {
            "en": "Classic Tailored White Thobe",
            "he": "ת'וב מחויט מסורתי",
            "ar": "ثوب أبيض خليجي مفصل",
            "es": "Thobe clásico entallado",
            "fr": "Thobe classique sur mesure",
            "de": "Klassisches maßgeschneidertes Thobe",
            "it": "Thobe classico su misura",
            "pt": "Thobe clássico sob medida",
            "nl": "Klassieke getailleerde thobe",
            "ru": "Классический сшитый на заказ тоб",
            "zh": "经典定制阿拉伯白长袍",
            "ja": "クラシックなテーラードソーブ",
            "hi": "क्लासिक सिलवाया हुआ थोब",
        },
        "shape": "crisp_column",
    },
    {
        "sub_category": "Abaya",
        "category": "Full Body",
        "item_type": "Modest Open-Front Abaya",
        "gender": "women",
        "dress_code": "smart-casual",
        "primary_fabrics": [("Crepe", 80), ("Silk", 20)],
        "patterns": ["solid", "embroidered"],
        "base_colors": [("Classic Black", (20, 20, 25)), ("Charcoal", (45, 45, 50)), ("Dusty Rose", (185, 140, 145))],
        "names": {
            "en": "Modest Crepe Embroidered Abaya",
            "he": "עבאיה צנועה רקומה מקרפ",
            "ar": "عباية كريب أنيقة ومطرزة",
            "es": "Abaya elegante de crepé bordada",
            "fr": "Abaya élégante en crêpe brodé",
            "de": "Elegante bestickte Krepp-Abaya",
            "it": "Abaya elegante in crêpe ricamato",
            "pt": "Abaya elegante de crepe bordada",
            "nl": "Elegante geborduurde crêpe abaya",
            "ru": "Элегантная креповая вышитая абайя",
            "zh": "优雅刺绣双绉阿巴亚长袍",
            "ja": "エレガントなクレープ刺繍アバヤ",
            "hi": "सुरुचिपूर्ण क्रेप कढ़ाईदार अबाया",
        },
        "shape": "cloak_open",
    },
    {
        "sub_category": "Kurta",
        "category": "Top",
        "item_type": "Mandarin Collar Kurta",
        "gender": "men",
        "dress_code": "smart-casual",
        "primary_fabrics": [("Linen", 60), ("Cotton", 40)],
        "patterns": ["solid", "embroidered", "printed"],
        "base_colors": [("Off-White", (240, 238, 230)), ("Saffron Yellow", (225, 160, 45)), ("Sky Blue", (90, 145, 210))],
        "names": {
            "en": "Linen Mandarin Collar Kurta",
            "he": "חולצת קורטה מפשתן עם צווארון סיני",
            "ar": "كورتا كتان بياقة ماندرين",
            "es": "Kurta de lino con cuello mao",
            "fr": "Kurta en lin à col mao",
            "de": "Leinen-Kurta mit Stehkragen",
            "it": "Kurta in lino con colletto alla coreana",
            "pt": "Kurta de linho com gola mandarim",
            "nl": "Linnen kurta met opstaande kraag",
            "ru": "Льняная курта с воротником-стойкой",
            "zh": "立领亚麻库尔塔长衫",
            "ja": "スタンドカラーのリネンクルタ",
            "hi": "लिनन मंदारिन कॉलर कुर्ता",
        },
        "shape": "tunic_slit",
    },
    {
        "sub_category": "Sherwani",
        "category": "Outerwear",
        "item_type": "Embroidered Royal Sherwani",
        "gender": "men",
        "dress_code": "formal",
        "primary_fabrics": [("Brocade", 65), ("Silk", 35)],
        "patterns": ["embroidered", "geometric"],
        "base_colors": [("Ivory Gold", (230, 215, 175)), ("Deep Maroon", (90, 15, 30)), ("Midnight Navy", (15, 25, 60))],
        "names": {
            "en": "Royal Brocade Wedding Sherwani",
            "he": "שרוואני חגיגי רקום מברוקאד",
            "ar": "شرواني زفاف فاخر مطرز",
            "es": "Sherwani de boda en brocado bordado",
            "fr": "Sherwani de mariage en brocart brodé",
            "de": "Hochzeits-Sherwani aus besticktem Brokat",
            "it": "Sherwani da cerimonia in broccato ricamato",
            "pt": "Sherwani de casamento em brocado bordado",
            "nl": "Koninklijke bruilofts-sherwani van brokaat",
            "ru": "Свадебный шервани из вышитой парчи",
            "zh": "奢华织锦刺绣谢瓦尼礼服外套",
            "ja": "ブロード刺繍ロイヤルシェルワニ",
            "hi": "शाही ब्रोकेड शादी शेरवानी",
        },
        "shape": "coat_mandarin",
    },
    {
        "sub_category": "Sari",
        "category": "Full Body",
        "item_type": "Banarasi Silk Sari",
        "gender": "women",
        "dress_code": "formal",
        "primary_fabrics": [("Silk", 85), ("Metallic Fibres", 15)],
        "patterns": ["embroidered", "floral", "geometric"],
        "base_colors": [("Crimson Red", (185, 20, 35)), ("Royal Magenta", (160, 25, 95)), ("Peacock Teal", (15, 110, 125))],
        "names": {
            "en": "Banarasi Silk Embroidered Sari",
            "he": "סארי משי בנרסי רקום",
            "ar": "ساري هندي حريري مطرز بالزري",
            "es": "Sari banarasi de seda bordado",
            "fr": "Sari en soie de Bénarès brodé",
            "de": "Banarasi-Seidensari mit Goldstickerei",
            "it": "Sari banarasi in seta ricamata",
            "pt": "Sári banarasi de seda bordada",
            "nl": "Banarasi zijden sari met gouddraad",
            "ru": "Шелковое сари Банараси с вышивкой",
            "zh": "瓦拉纳西丝绸刺绣萨丽",
            "ja": "バラナシシルク刺繍サリー",
            "hi": "बनारसी रेशमी कशीदाकारी साड़ी",
        },
        "shape": "draped_pallu",
    },
    {
        "sub_category": "Hanbok",
        "category": "Full Body",
        "item_type": "Traditional Korean Hanbok",
        "gender": "women",
        "dress_code": "formal",
        "primary_fabrics": [("Silk", 80), ("Ramie", 20)],
        "patterns": ["solid", "embroidered"],
        "base_colors": [("Pastel Pink & Jade", (235, 175, 185)), ("Cerulean & Coral", (80, 130, 195))],
        "names": {
            "en": "Traditional Korean Silk Hanbok",
            "he": "הנבוק קוריאני מסורתי ממשי",
            "ar": "هانبوك كوري تقليدي من الحرير",
            "es": "Hanbok coreano tradicional de seda",
            "fr": "Hanbok coréen traditionnel en soie",
            "de": "Traditioneller koreanischer Seiden-Hanbok",
            "it": "Hanbok coreano tradizionale in seta",
            "pt": "Hanbok coreano tradicional de seda",
            "nl": "Traditionele Koreaanse zijden hanbok",
            "ru": "Традиционный корейский шелковый ханбок",
            "zh": "传统韩国丝绸韩服",
            "ja": "伝統的な韓国の絹のチマチョゴリ（韓服）",
            "hi": "पारंपरिक कोरियाई रेशमी हानबोक",
        },
        "shape": "hanbok_chima",
    },
    {
        "sub_category": "Kimono",
        "category": "Full Body",
        "item_type": "Formal Silk Kimono with Obi",
        "gender": "women",
        "dress_code": "formal",
        "primary_fabrics": [("Silk", 90), ("Satin", 10)],
        "patterns": ["floral", "embroidered", "printed"],
        "base_colors": [("Cherry Blossom Indigo", (40, 50, 95)), ("Vermilion Orange", (215, 65, 40)), ("Mint Celadon", (150, 190, 170))],
        "names": {
            "en": "Formal Silk Kimono with Obi Sash",
            "he": "קימונו משי רשמי עם חגורת אובי",
            "ar": "كيمونو حريري رسمي مع حزام أوبي",
            "es": "Kimono formal de seda con faja obi",
            "fr": "Kimono formel en soie avec ceinture obi",
            "de": "Formeller Seidenkimono mit Obi-Gürtel",
            "it": "Kimono formale in seta con fascia obi",
            "pt": "Quimono formal de seda com faixa obi",
            "nl": "Formele zijden kimono met obi-band",
            "ru": "Формальное шелковое кимоно с поясом оби",
            "zh": "带腰带的和服正装",
            "ja": "帯付きの正装用絹着物",
            "hi": "ओबी बेल्ट के साथ रेशमी किमोनो",
        },
        "shape": "kimono_t",
    },
    {
        "sub_category": "Dirndl",
        "category": "Full Body",
        "item_type": "Bavarian Trachten Dirndl",
        "gender": "women",
        "dress_code": "smart-casual",
        "primary_fabrics": [("Cotton", 70), ("Linen", 30)],
        "patterns": ["floral", "plaid", "embroidered"],
        "base_colors": [("Alpine Forest Green", (45, 80, 50)), ("Berry Red", (155, 30, 50)), ("Bavarian Blue", (50, 90, 160))],
        "names": {
            "en": "Bavarian Folk Dirndl Dress with Apron",
            "he": "שמלת דירנדל בווארית מסורתית עם סינר",
            "ar": "فستان ديرندل بافاري تقليدي مع مريلة",
            "es": "Vestido bávaro Dirndl tradicional con delantal",
            "fr": "Robe traditionnelle bavaroise Dirndl avec tablier",
            "de": "Traditionelles bayerisches Trachten-Dirndl mit Schürze",
            "it": "Abito tradizionale bavarese Dirndl con grembiule",
            "pt": "Vestido bávaro tradicional Dirndl com avental",
            "nl": "Traditionele Beierse dirndl jurk met schort",
            "ru": "Традиционный баварский дирндль с передником",
            "zh": "带围裙的传统巴伐利亚迪恩德尔连衣裙",
            "ja": "エプロン付きの伝統的なバイエルン・ディアンドル",
            "hi": "पारंपरिक बवेरियन डिर्नड्ल ड्रेस एप्रन के साथ",
        },
        "shape": "dirndl_bodice",
    },
    {
        "sub_category": "Guayabera",
        "category": "Top",
        "item_type": "Four-Pocket Linen Guayabera",
        "gender": "men",
        "dress_code": "smart-casual",
        "primary_fabrics": [("Linen", 100)],
        "patterns": ["embroidered", "solid"],
        "base_colors": [("Natural Linen White", (245, 245, 242)), ("Pastel Sky", (140, 185, 220)), ("Sage Green", (130, 155, 135))],
        "names": {
            "en": "Linen Four-Pocket Guayabera Shirt",
            "he": "חולצת גואיאברה מפשתן עם ארבעה כיסים",
            "ar": "قميص غوايابيرا كتان بأربعة جيوب",
            "es": "Guayabera clásica de lino de cuatro bolsillos",
            "fr": "Chemise guayabera en lin à quatre poches",
            "de": "Leinen-Guayabera-Hemd mit vier Taschen",
            "it": "Camicia guayabera in lino a quattro tasche",
            "pt": "Camisa guayabera de linho com quatro bolsos",
            "nl": "Linnen guayabera-overhemd met vier zakken",
            "ru": "Льняная рубашка гуаябера с четырьмя карманами",
            "zh": "四口袋亚麻瓜亚贝拉衬衫",
            "ja": "4ポケット付きのリネン・グアジャベーラシャツ",
            "hi": "चार जेब वाली लिनन गुआयाबेरा शर्ट",
        },
        "shape": "guayabera_shirt",
    },
]

# Helper to render clean visual silhouette / texture reference images
def render_garment_image(filepath: Path, shape: str, rgb: tuple, pattern: str):
    img = Image.new("RGB", (448, 448), color=(240, 242, 245))
    draw = ImageDraw.Draw(img)

    # Garment main body coordinates
    if shape in ("long_tunic", "flared_robe", "crisp_column", "cloak_open"):
        # Full body long robe
        body = [(140, 60), (308, 60), (348, 410), (100, 410)]
        draw.polygon(body, fill=rgb, outline=(30, 30, 30), width=2)
        # Sleeves
        draw.polygon([(140, 60), (60, 190), (100, 210), (160, 120)], fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(308, 60), (388, 190), (348, 210), (288, 120)], fill=rgb, outline=(30, 30, 30), width=2)
        # Neckline
        draw.polygon([(190, 60), (224, 110), (258, 60)], fill=(240, 242, 245), outline=(30, 30, 30), width=2)
    elif shape in ("tunic_slit", "guayabera_shirt"):
        # Top tunic
        body = [(130, 70), (318, 70), (330, 340), (118, 340)]
        draw.polygon(body, fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(130, 70), (50, 180), (90, 200), (150, 120)], fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(318, 70), (398, 180), (358, 200), (298, 120)], fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(194, 70), (224, 115), (254, 70)], fill=(240, 242, 245), outline=(30, 30, 30), width=2)
    elif shape == "coat_mandarin":
        # Structured Sherwani
        body = [(140, 50), (308, 50), (340, 380), (108, 380)]
        draw.polygon(body, fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(140, 50), (60, 175), (95, 195), (155, 110)], fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(308, 50), (388, 175), (353, 195), (293, 110)], fill=rgb, outline=(30, 30, 30), width=2)
        draw.rectangle([(210, 50), (238, 75)], fill=rgb, outline=(30, 30, 30), width=2)
        draw.line([(224, 75), (224, 380)], fill=(20, 20, 20), width=3)
    else:
        # Sari / Hanbok / Kimono / Dirndl
        body = [(150, 60), (298, 60), (360, 420), (88, 420)]
        draw.polygon(body, fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(150, 60), (70, 170), (105, 190), (170, 110)], fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(298, 60), (378, 170), (343, 190), (278, 110)], fill=rgb, outline=(30, 30, 30), width=2)
        # Sash / Belt / Pallu
        draw.rectangle([(130, 180), (318, 220)], fill=(210, 170, 50), outline=(20, 20, 20), width=2)

    # Overlay embroidery thread markings if pattern is embroidered
    if pattern == "embroidered":
        gold_thread = (220, 180, 50)
        for y in range(90, 380, 24):
            draw.arc([204, y, 244, y + 16], start=0, end=180, fill=gold_thread, width=2)
            draw.line([(180, y), (200, y)], fill=gold_thread, width=2)
            draw.line([(248, y), (268, y)], fill=gold_thread, width=2)

    img.save(filepath, "JPEG", quality=90)

# Generate ~1,200 Balanced Multilingual Cultural Attire Records
cultural_samples = []
item_idx = 0
SYSTEM_PROMPT = (
    "Output raw JSON only ({...}). No markdown/intro.\n"
    "• Language: English keys/values/fabrics/colors.\n"
    "• name & title: 2-4 English words [Color] [Material/Cut] [Type]. Never generic ('Garment','Clothing').\n"
    "• category: 'Top'|'Bottom'|'Outerwear'|'Full Body'|'Footwear'|'Accessories'. Sweaters/tees='Top'. Outerwear=coats/jackets. Dresses/sets/robes/gowns='Full Body'.\n"
    "• sub_category: Specific cut ('T-Shirt','Sweater','Jeans','Pants','Skirt','Dresses','Suits','Galabiya','Kaftan','Thobe','Abaya','Kurta','Sherwani','Sari','Hanbok','Kimono','Dirndl','Guayabera','Oxfords','Loafers','Boots','Sandals','Sneakers','Heels','Pumps','Flats','Handbag','Sunglasses','Belts').\n"
    "• Cultural & Traditional: Galabiya/Kaftan/Thobe/Abaya/Sari/Hanbok/Kimono/Dirndl -> category:'Full Body' (open Abaya/Kaftan -> 'Outerwear'). Kurta/Guayabera -> 'Top'. Sherwani -> 'Outerwear'|'Top'.\n"
    "• pattern: 'solid'|'printed'|'geometric'|'striped'|'plaid'|'floral'|'camouflage'|'embroidered'."
)

ALL_LANGS = ["en", "he", "ar", "es", "fr", "de", "it", "pt", "nl", "ru", "zh", "ja", "hi"]

print('Generating high-fidelity cultural attire training pairs across all 13 languages...')
for arch in CULTURAL_ARCHETYPES:
    for color_name, rgb in arch["base_colors"]:
        for pat in arch["patterns"]:
            # Generate reference image
            img_filename = f"cultural_{arch['sub_category'].lower()}_{color_name.lower().replace(' ', '_')}_{pat}.jpg"
            img_path = CULTURAL_IMG_DIR / img_filename
            render_garment_image(img_path, arch["shape"], rgb, pat)

            for lang in ALL_LANGS:
                item_idx += 1
                name_localized = arch["names"].get(lang, arch["names"]["en"])
                clean_color_name = color_name.split()[-1]

                ground_truth = {
                    "category": arch["category"],
                    "sub_category": arch["sub_category"],
                    "item_type": arch["item_type"],
                    "name": f"{clean_color_name} {arch['sub_category']}",
                    "title": name_localized,
                    "caption": f"A {clean_color_name.lower()} {arch['sub_category'].lower()} crafted with {pat} {arch['primary_fabrics'][0][0].lower()}.",
                    "dress_code": arch["dress_code"],
                    "gender": arch["gender"],
                    "pattern": pat,
                    "colors": [{"name": clean_color_name, "pct": 100}],
                    "fabric_materials": [{"name": f[0], "pct": f[1]} for f in arch["primary_fabrics"]],
                    "season": ["spring", "summer"] if arch["dress_code"] != "formal" else ["fall", "winter", "spring"],
                    "tags": [arch["sub_category"].lower(), "traditional", pat, arch["dress_code"]],
                    "condition": "excellent",
                    "quality": "premium",
                    "state": "new",
                    "image_quality_status": "complete",
                    "image_quality_reason": None,
                    "reconstruction_prompt": None
                }

                user_prompt = (
                    f"**OUTPUT LANGUAGE = {lang}.** Extract structured garment attributes. Return raw JSON."
                )

                cultural_samples.append({
                    "image_path": str(img_path),
                    "system": SYSTEM_PROMPT,
                    "user": user_prompt,
                    "assistant": json.dumps(ground_truth, ensure_ascii=False),
                    "lang": lang,
                    "category": arch["category"],
                    "sub_category": arch["sub_category"]
                })

print(f'✅ Synthesized {len(cultural_samples)} cultural and religious attire training instances!')

# Combine baseline everyday garments with cultural garments
combined_dataset = baseline_samples + cultural_samples
random.seed(42)
random.shuffle(combined_dataset)

out_file = 'curated_dataset_v5.jsonl'
with open(out_file, 'w', encoding='utf-8') as f:
    for s in combined_dataset:
        f.write(json.dumps(s, ensure_ascii=False) + '\n')

print(f'🎉 Exported {len(combined_dataset)} total samples to {out_file} ({Path(out_file).stat().st_size / 1024 / 1024:.2f} MB)!')
print(f'• Baseline Western Garments: {len(baseline_samples)}')
print(f'• Cultural & Traditional Garments: {len(cultural_samples)}')


## 🚀 Step 4: Fine-Tuning Qwen2.5-VL-3B with QLoRA & Prompt Loss Masking
Fine-tune `Qwen/Qwen2.5-VL-3B-Instruct` using 4-bit BitsAndBytes quantization and PEFT LoRA.
- Native dynamic resolution processor (`min_pixels = 128*28*28`, `max_pixels = 448*448`).
- Prompt loss masking (`labels[:prompt_len] = -100`) ensures gradient updates strictly optimize the structured JSON output.
- Target attention projection matrices (`q_proj`, `k_proj`, `v_proj`, `o_proj`, dimension 2048) for optimal CUDA kernel speedup without BitsAndBytes unaligned MLP kernel stalls.
- Preserves the frozen vision encoder while fine-tuning multimodal alignment and multilingual output fluency.


In [ ]:
import json
from PIL import Image
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

# Verify GPU
if not torch.cuda.is_available():
    raise RuntimeError('❌ No GPU detected! In Colab, go to menu: Runtime -> Change runtime type -> Hardware accelerator: T4 or A100 GPU')

print(f'Loading Qwen2.5-VL processor for {MODEL_ID}...')
min_pixels = 128 * 28 * 28
max_pixels = 448 * 448
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=min_pixels, max_pixels=max_pixels)

# 4-bit NF4 Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Loading Qwen2.5-VL-3B-Instruct base model in 4-bit with SDPA...')
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation='sdpa',
    device_map='auto',
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)

# LoRA Configuration targeting attention projections
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()
model.print_trainable_parameters()

# Dataset with ChatML formatting and exact Prompt Loss Masking
class QwenCulturalEyesDataset(torch.utils.data.Dataset):
    def __init__(self, jsonl_file, processor, max_samples=None):
        self.items = []
        with open(jsonl_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    self.items.append(json.loads(line))
        if max_samples:
            self.items = self.items[:max_samples]
        self.processor = processor
        self.assistant_marker = processor.tokenizer.encode('<|im_start|>assistant\n', add_special_tokens=False)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        img_path = item['image_path']

        messages = [
            {'role': 'system', 'content': item['system']},
            {
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': img_path},
                    {'type': 'text', 'text': item['user']},
                ],
            },
            {'role': 'assistant', 'content': item['assistant']},
        ]

        text_full = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        image_inputs, video_inputs = process_vision_info(messages)

        full_inputs = self.processor(
            text=[text_full],
            images=image_inputs,
            videos=video_inputs,
            padding=False,
            return_tensors='pt',
        )

        input_ids = full_inputs['input_ids'].squeeze(0)
        labels = input_ids.clone()

        # Find the <|im_start|>assistant marker\n        input_list = input_ids.tolist()
        m_len = len(self.assistant_marker)
        prompt_boundary = len(input_list)
        for i in range(len(input_list) - m_len, -1, -1):
            if input_list[i : i + m_len] == self.assistant_marker:
                prompt_boundary = i + m_len
                break

        # Mask prompt & vision tokens (-100)
        labels[:prompt_boundary] = -100

        result = {k: v.squeeze(0) for k, v in full_inputs.items()}
        result['labels'] = labels
        return result

train_dataset = QwenCulturalEyesDataset('curated_dataset_v5.jsonl', processor, max_samples=6000)

# Batch collator
def collate_fn(batch):
    elem = batch[0]
    res = {}
    for k in elem:
        if k in ('input_ids', 'labels', 'attention_mask'):
            tensors = [d[k] for d in batch]
            pad_val = -100 if k == 'labels' else processor.tokenizer.pad_token_id
            res[k] = torch.nn.utils.rnn.pad_sequence(tensors, batch_first=True, padding_value=pad_val)
        elif isinstance(elem[k], torch.Tensor):
            res[k] = torch.stack([d[k] for d in batch])
        else:
            res[k] = [d[k] for d in batch]
    return res

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

training_args = TrainingArguments(
    output_dir='output_qwen_eyes_lora_v5',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=20,
    max_steps=800,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    warmup_steps=50,
    dataloader_num_workers=0,
    report_to='none',
)

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    args=training_args,
)

print('Starting QLoRA Fine-Tuning on Qwen2.5-VL-3B (Cultural Attire + Everyday Garments)...')
print('⏱️ Expected duration: ~25-35 minutes (800 steps)')
try:
    trainer.train()
    print('✅ Fine-tuning completed successfully!')
except KeyboardInterrupt:
    print('\n⚠️ Training interrupted early by user. Saving current checkpoint...')
finally:
    out_adapter = Path('output_qwen_eyes_lora_v5/adapter')
    out_adapter.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_adapter)
    processor.save_pretrained(out_adapter)
    print(f'✅ LoRA v5 Adapter successfully saved to {out_adapter}!')


## ⚙️ Step 5: Merge Adapter & Quantize to GGUF
Merge the fine-tuned LoRA v5 adapter into the base `Qwen2.5-VL-3B-Instruct` model and quantize with `llama.cpp` to `Q4_K_M` (production target).


In [ ]:
import os, sys
from pathlib import Path
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

# Bypass incompatible torchao check in PEFT
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
adapter_dir = 'output_qwen_eyes_lora_v5/adapter'
merged_dir = 'qwen2_5_vl_3b_lora_v5_merged'

print(f"Loading base model in FP16 for merging from {MODEL_ID}...")
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='cpu',
)

print(f"Merging LoRA adapter from {adapter_dir}...")
merged_model = PeftModel.from_pretrained(base_model, adapter_dir)
merged_model = merged_model.merge_and_unload()

print(f"Saving merged FP16 model to {merged_dir}...")
merged_model.save_pretrained(merged_dir)
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.save_pretrained(merged_dir)
print("✅ Merged model successfully saved!")

# Convert to GGUF using llama.cpp
print("🔄 Converting merged model to GGUF (FP16)...")
os.system(f"python llama.cpp/convert_hf_to_gguf.py {merged_dir} --outfile Qwen2.5-VL-3B-Instruct-v5-F16.gguf --outtype f16")

# Quantize to Q4_K_M
print("⚙️ Quantizing to Q4_K_M (production target)...")
os.system("./llama.cpp/build/bin/llama-quantize Qwen2.5-VL-3B-Instruct-v5-F16.gguf Qwen2.5-VL-3B-Instruct-v5-Q4_K_M.gguf Q4_K_M")

# Prepare production name
os.system("cp Qwen2.5-VL-3B-Instruct-v5-Q4_K_M.gguf Qwen2.5-VL-3B-Instruct-Q4_K_M.gguf")

gguf_path = Path("Qwen2.5-VL-3B-Instruct-Q4_K_M.gguf")
if gguf_path.exists():
    print(f"🎉 Production GGUF successfully built: {gguf_path.name} ({gguf_path.stat().st_size / 1024 / 1024:.1f} MB)")
else:
    raise RuntimeError("❌ GGUF quantization failed!")


## 🧪 Step 6: Cultural Attire Evaluation & Smoke Tests
Verify that the fine-tuned DressApp Eyes model accurately extracts:
1. `category: "Full Body"` (or `Top`/`Outerwear`) and exact cultural `sub_category` (`Galabiya`, `Kaftan`, `Thobe`, `Abaya`, `Kimono`, `Sari`, etc.).
2. Accurate recognition of `pattern: "embroidered"` and authentic fabric materials.
3. Idiomatic localized captions in Hebrew, Arabic, Hindi, Japanese, and Spanish.


In [ ]:
import json, glob
from PIL import Image
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

print('🧪 Loading merged model for evaluation...')
eval_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    'qwen2_5_vl_3b_lora_v5_merged',
    torch_dtype=torch.float16,
    device_map='auto',
)
processor = AutoProcessor.from_pretrained('qwen2_5_vl_3b_lora_v5_merged')

# Select test images across cultural archetypes
test_cases = [
    ('cultural_dataset/images/cultural_galabiya_white_embroidered.jpg', 'Hebrew (עברית)', 'Galabiya with embroidery'),
    ('cultural_dataset/images/cultural_kaftan_emerald_green_embroidered.jpg', 'Arabic (العربية)', 'Moroccan Kaftan in Arabic'),
    ('cultural_dataset/images/cultural_thobe_crisp_white_solid.jpg', 'English', 'White Thobe'),
    ('cultural_dataset/images/cultural_kimono_cherry_blossom_indigo_floral.jpg', 'Japanese (日本語)', 'Kimono in Japanese'),
    ('cultural_dataset/images/cultural_sari_crimson_red_embroidered.jpg', 'Hindi (हिन्दी)', 'Banarasi Sari in Hindi'),
    ('cultural_dataset/images/cultural_guayabera_natural_linen_white_embroidered.jpg', 'Spanish (Español)', 'Guayabera in Spanish'),
]

SYSTEM_PROMPT = (
    "Output raw JSON only ({...}). No markdown/intro.\n"
    "• Language: English keys/values/fabrics/colors.\n"
    "• name & title: 2-4 English words [Color] [Material/Cut] [Type]. Never generic ('Garment','Clothing').\n"
    "• category: 'Top'|'Bottom'|'Outerwear'|'Full Body'|'Footwear'|'Accessories'. Sweaters/tees='Top'. Outerwear=coats/jackets. Dresses/sets/robes/gowns='Full Body'.\n"
    "• sub_category: Specific cut ('T-Shirt','Sweater','Jeans','Pants','Skirt','Dresses','Suits','Galabiya','Kaftan','Thobe','Abaya','Kurta','Sherwani','Sari','Hanbok','Kimono','Dirndl','Guayabera','Oxfords','Loafers','Boots','Sandals','Sneakers','Heels','Pumps','Flats','Handbag','Sunglasses','Belts').\n"
    "• pattern: 'solid'|'printed'|'geometric'|'striped'|'plaid'|'floral'|'camouflage'|'embroidered'."
)

for img_path, lang_name, label in test_cases:
    if not os.path.exists(img_path):
        continue
    print(f'\n============================================================')
    print(f'🔍 Test: {label} (Output Language: {lang_name})')
    print(f'============================================================')

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': img_path},
                {'type': 'text', 'text': f'**OUTPUT LANGUAGE: {lang_name}**. Return raw JSON.'},
            ],
        },
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, return_tensors='pt').to(eval_model.device)

    with torch.no_grad():
        out = eval_model.generate(**inputs, max_new_tokens=400, temperature=0.1, do_sample=False)

    gen_text = processor.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(gen_text)
    try:
        parsed = json.loads(gen_text)
        print(f"✅ Extracted Sub-Category: {parsed.get('sub_category')}")
        print(f"✅ Extracted Pattern: {parsed.get('pattern')}")
        print(f"✅ Extracted Title: {parsed.get('title')}")
    except Exception as e:
        print(f"JSON Parse info: {e}")


## 🚀 Step 7: 1-Click Secure Push to Hetzner VPS Server
Upload the fine-tuned `Qwen2.5-VL-3B-Instruct-Q4_K_M.gguf` to `/srv/AI-Stylist/eyes_gguf/` on the Hetzner VPS (`178.105.144.142`) and restart the `dressapp-eyes` container.


In [ ]:
import os
import subprocess
from pathlib import Path

gguf_file = 'Qwen2.5-VL-3B-Instruct-Q4_K_M.gguf'
assert Path(gguf_file).exists(), f'❌ {gguf_file} not found!'

# 1. Test SSH connection
print('Testing SSH connection to Hetzner VPS (178.105.144.142)...')
try:
    proc = subprocess.run(
        ['ssh', '-o', 'BatchMode=yes', '-o', 'ConnectTimeout=5', '-o', 'StrictHostKeyChecking=no', 'root@178.105.144.142', 'echo connected'],
        capture_output=True, text=True, timeout=8
    )
    is_connected = ('connected' in proc.stdout)
except Exception:
    is_connected = False

if is_connected:
    print('✅ SSH Key authorized! Uploading directly to VPS at datacenter speeds...')
    os.system(f'scp -o StrictHostKeyChecking=no {gguf_file} root@178.105.144.142:/srv/AI-Stylist/eyes_gguf/')
    print('🔄 Restarting dressapp-eyes container...')
    os.system("ssh -o StrictHostKeyChecking=no root@178.105.144.142 'cd /srv/AI-Stylist/deploy && docker compose restart eyes && sleep 8 && docker exec dressapp-eyes curl -s http://127.0.0.1:7860/healthz'")
    print('🎉 Model successfully deployed and healthy!')
else:
    print('ℹ️ Colab SSH Key is not authorized on the VPS.')
    print('You can download the GGUF file directly to your local computer:')
    try:
        from google.colab import files
        files.download(gguf_file)
        print('⬇️ Triggered browser download!')
    except Exception as e:
        print(f'Manual download helper: {e}')
